In [97]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [98]:
class ActivationFunction:
   
    def __init__(self, AF) -> None:
        AF_der = f"{AF}_derivative"
        self.af_obj = getattr(self, AF, self.sigmoid)
        self.af_derivative_obj = getattr(self, AF_der, self.sigmoid_derivative)
    
    def af(self, x):
        return self.af_obj(x)
    
    def af_derivative(self, x):
        return self.af_derivative_obj(x)
        

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        return self.sigmoid(x) * (1 - self.sigmoid(x))
    
    def tanh(self, x):
        return np.tanh(x)
    
    def tanh_derivative(self, x):
        return 1 - np.tanh(x)**2

In [99]:
class NN:
    af = None
    def __init__(self) -> None:
        self.af = ActivationFunction('sigmoid')

    def network_initialize(self, input_features, n_hiddens=[3], output_features=1) -> None:
        np.random.seed(42)  # For reproducibility
        network = []
        layer_input = input_features
        for num_hidden in n_hiddens:
            weights = np.random.uniform(-1, 1, size=(layer_input + 1, num_hidden))
            network.append({'weights': weights})
            layer_input = num_hidden
        output_weights = np.random.uniform(-1, 1, size=(layer_input + 1, output_features))
        network.append({'weights': output_weights})
        return network
    
    def feedforward(self, network, X):
        net = np.dot(X, network[0]['weights'][:-1]) + network[0]['weights'][-1]
        network[0]['net'] = net
        network[0]['out'] = self.af.af(net)
        for i in range(1, len(network)):
            net = np.dot(network[i - 1]['out'], network[i]['weights'][:-1]) + network[i]['weights'][-1]
            network[i]['net'] = net
            network[i]['out'] = self.af.af(net)
        return network
    
    def BP_gradient(self, network, X, y, learning_rate):
        # Forward pass
        network = self.feedforward(network, X)
        
        # Calculate error for the output layer
        error = -2*(y - network[-1]['out'])
        delta_output = error * self.af.af_derivative(network[-1]['out'])
        # Update weights for the output layer

        network[-1]['weights'][:-1] -= learning_rate * np.dot(network[-2]['out'].T, delta_output)
        network[-1]['weights'][-1]  -= learning_rate * np.sum(delta_output, axis=0)  # Update bias
        
        # Calculate error for hidden layers and update weights
        for i in range(len(network) - 2, 0, -1):
            error = np.dot(delta_output, network[i + 1]['weights'][:-1].T)
            delta_hidden = error * self.af.af_derivative(network[i]['out'])
            network[i]['weights'][:-1] -= learning_rate * np.dot(network[i - 1]['out'].T, delta_hidden)
            network[i]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
            delta_output = delta_hidden
        
        # Update weights for the first hidden layer
        delta_output = delta_output
        error = np.dot(delta_output, network[1]['weights'][:-1].T)
        delta_hidden = error * self.af.af_derivative(network[0]['out'])
        network[0]['weights'][:-1] -= learning_rate * np.dot(X.T, delta_hidden)
        network[0]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
        return network

    def BP_train(self, x_train, y_train, x_test, y_test, epochs=1000, optimizer='gradient', learning_rate=0.001, activation_function='sigmoid', n_hiddens=[3, 3, 3]):
        
        self.af = ActivationFunction(activation_function)
        network = self.network_initialize(input_features=x_train.shape[1], n_hiddens=n_hiddens, output_features=y_train.shape[1])

        optimizer_obj = getattr(self, f"BP_{optimizer}",self.BP_gradient)

        for epoch in range(epochs):

            network = optimizer_obj(network, x_train, y_train, learning_rate)
            
            y_pred_test, y_pred_train, loss_test, loss_train = self.network_stats(network, x_train, y_train, x_test, y_test)

            self.print_result(epoch, y_train, y_test, y_pred_train, y_pred_test, loss_test, loss_train)

    def network_stats(self, network, x_train, y_train, x_test, y_test):
        y_pred_test = self.prediction(x_test, network)
        loss_test = np.mean(mean_squared_error(y_test, y_pred_test))


        y_pred_train = self.prediction(x_train, network)
        loss_train = np.mean(mean_squared_error(y_train, y_pred_train))
        return y_pred_test, y_pred_train, loss_test, loss_train


    def print_result(self, epoch, y_train, y_test, y_pred_train, y_pred_test, loss_test, loss_train):
        if not epoch % 50 == 0 : return
        print(f"epoch {epoch} finished", end=' | test_acc:')
        acc = accuracy_score(y_test, y_pred_test)
        print(acc, end=' | train_acc:')
        acc = accuracy_score(y_train, y_pred_train)
        print(acc, end=' | test_loss:')
        print(loss_test, end=' | train_loss:')
        print(loss_train)

    def prediction(self, X, trained_network):
        y_pred = np.round(self.feedforward(trained_network, X)[-1]['out'])
        return y_pred

In [100]:
def load_data(): 
    train_loaded = np.load(f'train_data.npz')
    test_loaded = np.load(f'test_data.npz')
    
    train_x, train_y = train_loaded['x'].T, train_loaded['y'] 
    test_x, test_y = test_loaded['x'].T, test_loaded['y'] 

    # Standarize data
    train_x = train_x / 255.
    test_x = test_x / 255.

    # Shuffle data (no effect for full batch method)
    train_indices = np.random.permutation(train_x.shape[1])
    train_x = train_x[:, train_indices]
    train_y = train_y[train_indices]

    test_indices = np.random.permutation(test_x.shape[1])
    test_x = test_x[:, test_indices]
    test_y = test_y[test_indices]

    return train_x.T, train_y.reshape(-1, 1), test_x.T, test_y.reshape(-1, 1)

train_x, train_y, test_x, test_y = load_data()
nn = NN()
trained_network = nn.BP_train(train_x, train_y, test_x, test_y, n_hiddens=[3,2])

epoch 0 finished | test_acc:0.5 | train_acc:0.5 | test_loss:0.5 | train_loss:0.5
epoch 50 finished | test_acc:0.5075 | train_acc:0.511 | test_loss:0.4925 | train_loss:0.489
epoch 100 finished | test_acc:0.7175 | train_acc:0.703 | test_loss:0.2825 | train_loss:0.297
epoch 150 finished | test_acc:0.63 | train_acc:0.641 | test_loss:0.37 | train_loss:0.359
epoch 200 finished | test_acc:0.7625 | train_acc:0.746 | test_loss:0.2375 | train_loss:0.254
epoch 250 finished | test_acc:0.8375 | train_acc:0.828 | test_loss:0.1625 | train_loss:0.172
epoch 300 finished | test_acc:0.8825 | train_acc:0.8815 | test_loss:0.1175 | train_loss:0.1185


KeyboardInterrupt: 